## Retrieving Episode Files with Beautiful Soup

This scraper is designed to retrieve transcripts of Spy X Family from <https://transcripts.foreverdreaming.org/viewforum.php?f=1524>. Dr. B started this scraper file to help stand in for a missing member of the student project team.

**Caution:** The DIGIT Code Mentors / Battletesters team found all 38 scripts from Seasons 1 and 2 (as of Spring 2026) on this site,
but they are NOT formatted all the same. There are two different kinds of formatting we saw:
1. **Season 1, Episodes 1 – 12** provides speaker names followed by their spoken text.
2. **Season 1 Episodes 1+ and all of Season 2** are NOT providing character names.

That said, all of this transcript material may be useful. Season 1 episodes 1–12, if converted to XML, could be processed with a network to show which speakers show up together in the episodes. And you could extract JUST the dialogue text from those files to make a complete uniform corpus of all episodes.

This notebook requires that you complete the following installations in your activated virtual environment for Beautiful Soup 4, playwright, and chromium (installed with playwright).
* `pip install bs4`.
* `pip install playwright`
* `playwright install chromium`


In [1]:
import bs4
import requests
import re  # this lets us do regex replacements
import os
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

In [ ]:
archive_url = "https://transcripts.foreverdreaming.org/viewforum.php?f=1524"   
output_folder = "../transcriptFiles"

#### Getting around 403 and security errors
Our first attempt to pull from the archive_url led to 403 errors. 
Read about how to avoid these by sending a requests header (see <https://scrape.do/blog/python-requests-403-forbidden/>).
This site is guarded against bots by a system called Anubis. In the next couple of cells, we're assuring the site that we're not malicious, 
and we're checking to make sure we're not being redirected (and we're not). 

We found we had to work with asyncio and the playwright library to help deliver requests from the archive site.

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
}
# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
# }

response = requests.get(archive_url, headers=headers)
print(response.status_code)  # Returns 200 if successful

In [ ]:
# print(response.status_code)   # Should be 200
# print(len(response.text))     # Compare to the page size you see in browser DevTools > Network tab
# print(response.text[:2000])
print(response.text)
# print(response.url)       # Did you end up somewhere different than you requested?
# print(response.history)   # Were there any redirects along the way?

In [ ]:
# DIAGNOSTIC TO MAKE SURE WE'RE SEEING LINKS AND LINK TEXT WHERE WE EXPECT

# In a Jupyter/IPython environment, you can run this directly in a cell:
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()
    
    # Replace with your actual URL
    await page.goto(archive_url)
    
    # Wait for the specific selector to ensure the page is loaded
    await page.wait_for_selector("a.topictitle", timeout=15000)
    
    html = await page.content()
    await browser.close()

# Now you can parse with BeautifulSoup as usual
soup = BeautifulSoup(html, "html.parser")
links = soup.find_all("a", class_="topictitle")

for link in links:
    print(link.get_text(), link.get('href'))

# ebb: NOTE: we pick up one extra link that isn't an episode transcript: 2026 ~ Secrets of the Board. We'll just manually remove it. 


In [ ]:
# Scraping Episodes
async def get_transcript_text(browser, url):
    page = await browser.new_page()
    await page.goto(url)
    
    # Wait for the main post body to load (Forever Dreaming uses .postbody)
    await page.wait_for_selector(".postbody", timeout=10000)
    
    html = await page.content()
    await page.close() # Close the tab, but keep the browser open
    
    soup = BeautifulSoup(html, "html.parser")
    # Forever Dreaming transcript text is inside div.content
    content = soup.find("div", class_="content")
    return content.get_text(separator="\n") if content else ""

# Launch the Scraper
async def corpus_build(archive_url, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        print(f"Fetching index page: {archive_url}")
        await page.goto(archive_url)
        await page.wait_for_selector("a.topictitle")
        
        html = await page.content()
        soup = BeautifulSoup(html, "html.parser")
        
        # Find all transcript links
        links = soup.find_all("a", class_="topictitle")
        
        for index, link in enumerate(links):
            raw_title = link.get_text().strip().replace("/", "-") # Here we replace `/` that could be a path separator 
            title = re.sub(r'[^\w\-]', '_', raw_title)   # We're cleaning up the title with a regex, 
             # anything that isn't alphanumeric gets converted to an underscore.
            relative_url = link.get('href').lstrip('.')
            full_url = f"https://transcripts.foreverdreaming.org{relative_url}"
            
            print(f"Scraping ({index+1}/{len(links)}): {title}")
            
            try:
                text = await get_transcript_text(browser, full_url)
                
                # Write to file
                file_path = os.path.join(output_folder, f"{title}.txt")
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(text)
            except Exception as e:
                print(f"Failed to scrape {title}: {e}")

        await browser.close()
        print("Finished! Corpus saved.")

# Initiate the process by running corpus_build()

await corpus_build(
    archive_url, 
    output_folder
)